In [17]:
# Cell 1 - project setup + imports + load baseline models

from pathlib import Path
import sys

# figure out project root and make src importable 
CWD = Path.cwd()
if (CWD / "src").exists():
    PROJECT_DIR = CWD
elif (CWD.parent / "src").exists():
    PROJECT_DIR = CWD.parent
else:
    PROJECT_DIR = CWD.parent.parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project dir:", PROJECT_DIR)

# core imports
import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

# local project imports 
from src.artifacts import (
    load_static_rf,
    load_static_xgb,
    load_behavior_model_and_vec,
)
from src.hybrid import (
    EarlyFusionModel,
    LateFusionScorer,
    GatedLateFusionScorer,
    concat_features,
)
from src.cape_data import build_cape_df, make_cape_text_splits

# model dir + baseline artefacts
MODEL_DIR = PROJECT_DIR / "saved" / "models"
print("Model dir:", MODEL_DIR)

drebin_rf = load_static_rf()
drebin_xgb = load_static_xgb()
cape_rf, cape_tfidf = load_behavior_model_and_vec()

print("Static RF:", type(drebin_rf))
print("Static XGB:", type(drebin_xgb))
print("Behaviour RF:", type(cape_rf))
print("Behaviour TF-IDF:", type(cape_tfidf))

Project dir: /Users/anmoulmalhotra/Documents/ProjectDissertationDir/hybrid-av
Model dir: /Users/anmoulmalhotra/Documents/ProjectDissertationDir/hybrid-av/saved/models
Static RF: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Static XGB: <class 'xgboost.sklearn.XGBClassifier'>
Behaviour RF: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Behaviour TF-IDF: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [7]:
#Build or load a dataset where each row has:
#  X_static_joint[i]   -> static-style features for sample i
#  X_behavior_joint[i] -> behaviour features for sample i
#  y_joint[i]          -> labels (same label space for both models)
#
#For example, for an initial experiment wemight:
#  * Use CAPE only, and define:
#     - X_static_like = TF-IDF with a very small vocabulary or using only imports
#     - X_behavior    = thefull TF-IDF (current behaviour features)
#  * Or build some other proxy that respects "same label space".
#
#here just create placeholders so the rest of the notebook runs once 
#populate them.

# Cell 2 – hybrid evaluation helper functions

def evaluate_late_fusion(
    static_model,
    behavior_model,
    Xs_train,
    Xb_train,
    y_train,
    Xs_val,
    Xb_val,
    y_val,
    class_order=None,
    alpha_grid=(0.0, 0.25, 0.5, 0.75, 1.0),
):
    """
    Scan alpha over a grid and return a DataFrame with validation metrics.
    """
    results = []
    for alpha in alpha_grid:
        scorer = LateFusionScorer(
            static_model=static_model,
            behavior_model=behavior_model,
            alpha=alpha,
            class_order=class_order,
        )
        y_val_pred = scorer.predict(Xs_val, Xb_val)
        acc = accuracy_score(y_val, y_val_pred)
        f1m = f1_score(y_val, y_val_pred, average="macro")
        results.append({"alpha": alpha, "val_accuracy": acc, "val_macro_f1": f1m})
    return pd.DataFrame(results)


import time

def timed_gated_predictions(
    scorer: GatedLateFusionScorer, X_static, X_behavior, use_gate=True
):
    """
    Measure total prediction time and fraction of samples that triggered
    the behaviour branch.
    """
    start = time.time()
    p_static = scorer._proba_in_order(
        scorer.static_model, X_static, scorer.class_order
    )
    max_conf = p_static.max(axis=1)
    need_beh = max_conf < scorer.static_conf_thresh

    if use_gate and np.any(need_beh):
        X_beh_uncertain = (
            X_behavior[need_beh]
            if not sparse.isspmatrix(X_behavior)
            else X_behavior[need_beh, :]
        )
        _ = scorer._proba_in_order(
            scorer.behavior_model, X_beh_uncertain, scorer.class_order
        )

    total_time = time.time() - start
    frac_behaviour = float(need_beh.mean())

    return total_time, frac_behaviour


def evaluate_early_fusion(
    Xs_train,
    Xb_train,
    y_train,
    Xs_val,
    Xb_val,
    y_val,
    base_clf=None,
):
    """
    Fit an EarlyFusionModel on [Xs | Xb] and compute VAL metrics.
    """
    model = EarlyFusionModel(base_clf=base_clf)
    model.fit(Xs_train, Xb_train, y_train)
    y_val_pred = model.predict(Xs_val, Xb_val)
    acc = accuracy_score(y_val, y_val_pred)
    f1m = f1_score(y_val, y_val_pred, average="macro")
    return model, acc, f1m

In [8]:
#build CAPE DataFrame and text splits

cape_df = build_cape_df(PROJECT_DIR, min_api_len=10)
print("CAPE DF shape:", cape_df.shape)
print("Families (top 10):")
print(cape_df["family"].value_counts().head(10))

X_train_text, X_val_text, X_test_text, y_train, y_val, y_test = make_cape_text_splits(
    cape_df, random_state=42
)

print("Train size:", len(X_train_text))
print("Val size  :", len(X_val_text))
print("Test size :", len(X_test_text))

CAPE DF shape: (48976, 5)
Families (top 10):
family
Emotet      14429
Swisyn      12591
Qakbot       4895
Trickbot     4202
Lokibot      4191
njRAT        3372
Zeus         2594
Ursnif       1343
Adload        704
HarHar        655
Name: count, dtype: int64
Train size: 29385
Val size  : 9795
Test size : 9796


In [9]:
# define cheap vs full TF-IDF views on CAPE

# Cheap / low-budget view: only unigrams, smaller vocab, drop rare tokens
tfidf_cheap = TfidfVectorizer(
    ngram_range=(1, 1),
    max_features=1000,
    min_df=10,
)

X_train_cheap = tfidf_cheap.fit_transform(X_train_text)
X_val_cheap = tfidf_cheap.transform(X_val_text)
X_test_cheap = tfidf_cheap.transform(X_test_text)

# Full behavioural view: richer TF-IDF (similar to behaviour baseline)
tfidf_full = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    min_df=5,
)

X_train_full = tfidf_full.fit_transform(X_train_text)
X_val_full = tfidf_full.transform(X_val_text)
X_test_full = tfidf_full.transform(X_test_text)

print("Cheap view train shape :", X_train_cheap.shape)
print("Full view train shape  :", X_train_full.shape)

Cheap view train shape : (29385, 1000)
Full view train shape  : (29385, 5000)


In [10]:
# train separate RFs on cheap vs full views

rf_cheap = RandomForestClassifier(
    n_estimators=200,
    max_depth=30,
    random_state=42,
    n_jobs=-1,
)
rf_full = RandomForestClassifier(
    n_estimators=200,
    max_depth=30,
    random_state=42,
    n_jobs=-1,
)

rf_cheap.fit(X_train_cheap, y_train)
rf_full.fit(X_train_full, y_train)

# Baseline validation metrics for each view
y_val_pred_cheap = rf_cheap.predict(X_val_cheap)
y_val_pred_full = rf_full.predict(X_val_full)

acc_cheap = accuracy_score(y_val, y_val_pred_cheap)
f1_cheap = f1_score(y_val, y_val_pred_cheap, average="macro")

acc_full = accuracy_score(y_val, y_val_pred_full)
f1_full = f1_score(y_val, y_val_pred_full, average="macro")

print(f"Cheap RF - val acc:  {acc_cheap:.4f}, macro-F1: {f1_cheap:.4f}")
print(f"Full RF  - val acc:  {acc_full:.4f}, macro-F1: {f1_full:.4f}")

Cheap RF - val acc:  0.9898, macro-F1: 0.9859
Full RF  - val acc:  0.9909, macro-F1: 0.9878


In [18]:
# late fusion on CAPE (cheap RF + full RF) over alpha grid

class_order = sorted(np.unique(y_train))

late_fusion_val_df = evaluate_late_fusion(
    static_model=rf_cheap,
    behavior_model=rf_full,
    Xs_train=X_train_cheap,
    Xb_train=X_train_full,
    y_train=y_train,
    Xs_val=X_val_cheap,
    Xb_val=X_val_full,
    y_val=y_val,
    class_order=class_order,
    alpha_grid=(0.0, 0.25, 0.5, 0.75, 1.0),
)

late_fusion_val_df

,alpha,val_accuracy,val_macro_f1
0,0.00,0.990914,0.987825
1,0.25,0.990403,0.986793
2,0.50,0.990097,0.986565
3,0.75,0.990097,0.986066
4,1.00,0.989791,0.985868


In [19]:
# pick best alpha and run gated late fusion with timing

best_row = late_fusion_val_df.sort_values("val_macro_f1").iloc[-1]
best_alpha = float(best_row["alpha"])
print("Best alpha on VAL (macro-F1):", best_alpha)

gated_scorer = GatedLateFusionScorer(
    static_model=rf_cheap,
    behavior_model=rf_full,
    alpha=best_alpha,
    class_order=class_order,
    static_conf_thresh=0.9,  # can tune later
)

total_time_val, frac_behaviour_val = timed_gated_predictions(
    scorer=gated_scorer,
    X_static=X_val_cheap,
    X_behavior=X_val_full,
    use_gate=True,
)

print(
    f"Gated late fusion (VAL): time={total_time_val:.3f}s, "
    f"behaviour used on {frac_behaviour_val*100:.1f}% of samples"
)

Best alpha on VAL (macro-F1): 0.0
Gated late fusion (VAL): time=0.112s, behaviour used on 6.8% of samples


In [20]:
#  Gated late fusion accuracy & macro-F1 on VAL

y_val_pred_gated = gated_scorer.predict(X_val_cheap, X_val_full)
acc_gated_val = accuracy_score(y_val, y_val_pred_gated)
f1_gated_val = f1_score(y_val, y_val_pred_gated, average="macro")

print(f"Gated late fusion (VAL) - acc: {acc_gated_val:.4f}, macro-F1: {f1_gated_val:.4f}")

Gated late fusion (VAL) - acc: 0.9909, macro-F1: 0.9878


In [16]:
# early fusion RF on concatenated [cheap | full] CAPE views

early_model, early_acc, early_f1 = evaluate_early_fusion(
    Xs_train=X_train_cheap,
    Xb_train=X_train_full,
    y_train=y_train,
    Xs_val=X_val_cheap,
    Xb_val=X_val_full,
    y_val=y_val,
)

print(f"Early fusion RF - val acc: {early_acc:.4f}, macro-F1: {early_f1:.4f}")

Early fusion RF - val acc: 0.9854, macro-F1: 0.9808


In [22]:
# --- Test-set metrics: cheap RF, full RF, late fusion, gated, early fusion ---

results_test = []

# 1) Cheap RF (static-like view)
y_test_pred_cheap = rf_cheap.predict(X_test_cheap)
acc_cheap_test = accuracy_score(y_test, y_test_pred_cheap)
f1_cheap_test = f1_score(y_test, y_test_pred_cheap, average="macro")
results_test.append(
    {
        "model": "cheap_rf",
        "split": "test",
        "accuracy": acc_cheap_test,
        "macro_f1": f1_cheap_test,
    }
)

# 2) Full RF (behaviour view)
y_test_pred_full = rf_full.predict(X_test_full)
acc_full_test = accuracy_score(y_test, y_test_pred_full)
f1_full_test = f1_score(y_test, y_test_pred_full, average="macro")
results_test.append(
    {
        "model": "full_rf",
        "split": "test",
        "accuracy": acc_full_test,
        "macro_f1": f1_full_test,
    }
)

# 3) Late fusion (no gate) with best alpha
late_fusion_test = LateFusionScorer(
    static_model=rf_cheap,
    behavior_model=rf_full,
    alpha=best_alpha,
    class_order=class_order,
)
y_test_pred_late = late_fusion_test.predict(X_test_cheap, X_test_full)
acc_late_test = accuracy_score(y_test, y_test_pred_late)
f1_late_test = f1_score(y_test, y_test_pred_late, average="macro")
results_test.append(
    {
        "model": f"late_fusion_alpha_{best_alpha}",
        "split": "test",
        "accuracy": acc_late_test,
        "macro_f1": f1_late_test,
    }
)

# 4) Gated late fusion (same best_alpha, same gate threshold)
y_test_pred_gated = gated_scorer.predict(X_test_cheap, X_test_full)
acc_gated_test = accuracy_score(y_test, y_test_pred_gated)
f1_gated_test = f1_score(y_test, y_test_pred_gated, average="macro")
results_test.append(
    {
        "model": f"gated_late_fusion_alpha_{best_alpha}_thresh_{gated_scorer.static_conf_thresh}",
        "split": "test",
        "accuracy": acc_gated_test,
        "macro_f1": f1_gated_test,
    }
)

# 5) Early fusion RF (concatenated [cheap | full])
y_test_pred_early = early_model.predict(X_test_cheap, X_test_full)
acc_early_test = accuracy_score(y_test, y_test_pred_early)
f1_early_test = f1_score(y_test, y_test_pred_early, average="macro")
results_test.append(
    {
        "model": "early_fusion_rf",
        "split": "test",
        "accuracy": acc_early_test,
        "macro_f1": f1_early_test,
    }
)

# Show a neat summary table
test_summary_df = pd.DataFrame(results_test)
test_summary_df

,model,split,accuracy,macro_f1
0,cheap_rf,test,0.988771,0.983126
1,full_rf,test,0.989281,0.984815
2,late_fusion_alpha_0.0,test,0.989281,0.984815
3,gated_late_fusion_alpha_0.0_thresh_0.9,test,0.989281,0.984815
4,early_fusion_rf,test,0.982850,0.977552


In [23]:
# --- Simple timing on TEST set for different inference strategies ---

def time_predict(model, X_static=None, X_behavior=None, mode="single"):
    """
    mode:
      - "static": model.predict on X_static
      - "behavior": model.predict on X_behavior
      - "late": LateFusionScorer.predict on (X_static, X_behavior)
      - "gated": GatedLateFusionScorer.predict on (X_static, X_behavior)
      - "early": EarlyFusionModel.predict on (X_static, X_behavior)
    """
    start = time.time()
    if mode == "static":
        _ = model.predict(X_static)
    elif mode == "behavior":
        _ = model.predict(X_behavior)
    elif mode == "late":
        _ = model.predict(X_static, X_behavior)
    elif mode == "gated":
        _ = model.predict(X_static, X_behavior)
    elif mode == "early":
        _ = model.predict(X_static, X_behavior)
    else:
        raise ValueError(f"Unknown mode: {mode}")
    total_time = time.time() - start
    return total_time


n_test = X_test_cheap.shape[0]

timings = []

# cheap RF
t_cheap = time_predict(rf_cheap, X_static=X_test_cheap, mode="static")
timings.append({"model": "cheap_rf", "split": "test", "total_s": t_cheap, "ms_per_sample": 1000 * t_cheap / n_test})

# full RF
t_full = time_predict(rf_full, X_behavior=X_test_full, mode="behavior")
timings.append({"model": "full_rf", "split": "test", "total_s": t_full, "ms_per_sample": 1000 * t_full / n_test})

# late fusion (no gate)
late_fusion_test = LateFusionScorer(
    static_model=rf_cheap,
    behavior_model=rf_full,
    alpha=best_alpha,
    class_order=class_order,
)
t_late = time_predict(late_fusion_test, X_static=X_test_cheap, X_behavior=X_test_full, mode="late")
timings.append({"model": f"late_fusion_alpha_{best_alpha}", "split": "test", "total_s": t_late, "ms_per_sample": 1000 * t_late / n_test})

# gated late fusion
t_gated, frac_behaviour_test = timed_gated_predictions(
    scorer=gated_scorer,
    X_static=X_test_cheap,
    X_behavior=X_test_full,
    use_gate=True,
)
timings.append(
    {
        "model": f"gated_late_fusion_alpha_{best_alpha}_thresh_{gated_scorer.static_conf_thresh}",
        "split": "test",
        "total_s": t_gated,
        "ms_per_sample": 1000 * t_gated / n_test,
        "behaviour_fraction": frac_behaviour_test,
    }
)

# early fusion
t_early = time_predict(early_model, X_static=X_test_cheap, X_behavior=X_test_full, mode="early")
timings.append({"model": "early_fusion_rf", "split": "test", "total_s": t_early, "ms_per_sample": 1000 * t_early / n_test})

timings_df = pd.DataFrame(timings)
timings_df

,model,split,total_s,ms_per_sample,behaviour_fraction
0,cheap_rf,test,0.072376,0.007388,NaN
1,full_rf,test,0.162543,0.016593,NaN
2,late_fusion_alpha_0.0,test,0.226194,0.023090,NaN
3,gated_late_fusion_alpha_0.0_thresh_0.9,test,0.084983,0.008675,0.066966
4,early_fusion_rf,test,0.267288,0.027285,NaN


In [24]:
from joblib import dump

HYBRID_MODEL_DIR = MODEL_DIR  # reuse saved/models/

# Save cheap TF-IDF + RF
dump(tfidf_cheap, HYBRID_MODEL_DIR / "cape_tfidf_cheap.pkl")
dump(rf_cheap, HYBRID_MODEL_DIR / "cape_rf_cheap.pkl")

# Save full TF-IDF + RF
dump(tfidf_full, HYBRID_MODEL_DIR / "cape_tfidf_full.pkl")
dump(rf_full, HYBRID_MODEL_DIR / "cape_rf_full.pkl")

# Save early fusion model as well, just for completeness
dump(early_model, HYBRID_MODEL_DIR / "cape_early_fusion_rf.pkl")

['/Users/anmoulmalhotra/Documents/ProjectDissertationDir/hybrid-av/saved/models/cape_early_fusion_rf.pkl']